# In this notebook, we will cleanup the data analysis given to us!

In [1]:
import pandas as pd
from pathlib import Path
import unicodedata
import re

raw_data = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/2026-02-06_LiveCPpilot__2026-02-06T13_25_31-Measurement 3/Evaluation20/Objects_Population - NoBorderObj.txt")
out_data = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/2026-02-06_LiveCPpilot__2026-02-06T13_25_31-Measurement 3/2026-02-06_LiveCPpilot_cleaned_data.txt")

In [2]:
def clean_column(name):
    replacements = {
        'µ': 'u',
        'μ': 'u',
        '²': '2',
        '³': '3',
    }
    
    for char, replacement in replacements.items():
        name = name.replace(char, replacement)
    
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("ascii")
    name = name.strip()
    name = name.strip('_')
    
    return name

let's remove the header and write to a clean output

In [25]:
with open(raw_data, "r", encoding="utf-8") as in_file,\
    open(out_data, "w", encoding="utf-8") as out_file:
    for i, line in enumerate(in_file):
        curr_line = line.strip()
        
        # print(f"{curr_line} :: {i}")
        
        if i < 9:
            print(f"{curr_line} :: {i}")
            continue
        if i == 9:
            cleaned = "\t".join(clean_column(col) for col in line.strip().split("\t"))
            out_file.write(cleaned + "\n")
        else:
            out_file.write(line)
            
        

Database Name	ODA :: 0
Database Link	http://localhost/ODA/OdaService.asmx :: 1
Evaluation Signature	f1523d9c-d3db-4542-a7b9-36350aa04472 :: 2
Plate Name	2026-02-06_LiveCPpilot :: 3
Measurement	Measurement 3 :: 4
Evaluation	Evaluation20 :: 5
Population	Population - NoBorderObj :: 6
 :: 7
[Data] :: 8


# We will create a function to properly process and clean the data

In [6]:
import pandas as pd
from pathlib import Path
import unicodedata
import re
import os
import shutil
from time import sleep

In [7]:
def clean_column(name):
    replacements = {
        'µ': 'u',
        'μ': 'u',
        '²': '2',
        '³': '3',
    }
    
    for char, replacement in replacements.items():
        name = name.replace(char, replacement)
    
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("ascii")
    name = name.strip()
    name = name.strip('_')
    
    return name

def parse_harmony_directory(root: Path) -> Path:
    """
    Parses a harmony file directory and yields the cell analysis file

    Args:
        root (Path): Top level folder where all the harmony datasets are located

    Yields:
        Path, str: yields a file path to the cell analysis AND the harmony dataset name
    
    **NOTE:** The harmony file structure is:
    
        dataset name/
        ├─ raw name/
        │  ├─ Evaluation/
        │  │  ├─ PlateResult.txt
        │  │  ├─ Objects_Population - NoBorderObj.txt (this is analysis file)
        │  │  ├─ .ExportDone_CellListsToTxt
        │  ├─ indexfile.txt

    """
    for f in root.glob("*/*/*/*"):
        if f.is_file() and not f.name.startswith(".") and f.name != "PlateResults.txt":
            yield f, f.relative_to(root).parts[0]
        

def clean_folder(path: str) -> None:
    if os.path.exists(path):
        shutil.rmtree(path)
        sleep(0.5)
    os.makedirs(path, exist_ok=True)

# in_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study")
# for f in parse_harmony_directory(root=in_path):
#     print(f)

In [9]:
in_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/harmony_datasets")
out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/formatted_raw_datasets/")

clean_folder(str(out_path))

for file, ds_name in parse_harmony_directory(in_path):
    
    print(ds_name)
    
    out_name = out_path / f"{ds_name}.tsv"
    with open(file, "r", encoding="utf-8") as in_file, open(out_name, "w", encoding="utf-8") as out_file:
        for i, line in enumerate(in_file):
            curr_line = line.strip()
        
            # print(f"{curr_line} :: {i}")
            
            if i < 9:
                print(f"{curr_line} :: {i}")
                continue
            if i == 9:
                cleaned = "\t".join(clean_column(col) for col in line.strip().split("\t"))
                out_file.write(cleaned + "\n")
            else:
                out_file.write(line.rstrip("\t\n") + "\n")
    
    print("\n\n")
    

2026-03-10CelPaint_ChromaNoTreatDex
Database Name	ODA :: 0
Database Link	http://localhost/ODA/OdaService.asmx :: 1
Evaluation Signature	f1523d9c-d3db-4542-a7b9-36350aa04472 :: 2
Plate Name	2026-02-06_LiveCPpilot :: 3
Measurement	Measurement 3 :: 4
Evaluation	Evaluation20 :: 5
Population	Population - NoBorderObj :: 6
 :: 7
[Data] :: 8



2026-03-11CelPaint_PhenovNoTreatDex
Database Name	CSC :: 0
Database Link	http://192.168.0.1/ODA/OdaService.asmx :: 1
Evaluation Signature	2af85bb2-c52a-4b44-8fde-f84bdb053344 :: 2
Plate Name	2026-02-06_LiveCPpilot :: 3
Measurement	Measurement 3 :: 4
Evaluation	Evaluation21 :: 5
Population	Population - NoBorderObj :: 6
 :: 7
[Data] :: 8



2026-03-11CelPaint_PhenovNoTreatDex_HARMONY_FIXED
Database Name	CSC :: 0
Database Link	http://192.168.0.1/ODA/OdaService.asmx :: 1
Evaluation Signature	5a8c21bf-fb31-4fc8-81c6-68351caa6014 :: 2
Plate Name	2026-02-06_LiveCPpilot :: 3
Measurement	Measurement 3 :: 4
Evaluation	Evaluation26 :: 5
Population	Population - NoB

In [10]:
# we need to get rid of one file (this is the old phenovue)

old_file = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/formatted_raw_datasets/2026-03-11CelPaint_PhenovNoTreatDex.tsv")
old_file.unlink()